# Prediction-Driven Browser Recommendations Engine

This notebook demonstrates the end-to-end **browser recommendation pipeline**.

### Recommendation Engine Methodology:
1. **LSTM Forecast Integration**: Ranks categories based on the LSTM next-session category probability forecast.
2. **Historical Affinity Fusion**: Combines predicted next categories with the user's historical category engagement frequency.
3. **Rule-Based Heuristics**: Evaluates productivity vs entertainment ratio, RAM usage threshold alerts, and session fragmentation.
4. **Severity Classification**: Tags suggestions with severity levels (`high` 🔴, `medium` 🟠, `low` 🟢).

## 1. Environment Setup & Pipeline Import

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from src.recommendation.pipeline import RecommendationPipeline
from src.utils.mlflow_utils import get_tracking_uri, setup_mlflow

setup_mlflow()
print(f"MLflow Tracking URI: {get_tracking_uri()}")

## 2. Execute Recommendation Pipeline

In [ ]:
pipeline = RecommendationPipeline()
recommendations_df = pipeline.run()

print(f"Generated {len(recommendations_df)} recommendations across test sessions.")
display(recommendations_df.head(15))

## 3. Analyze Recommendation Distribution & Severities

In [ ]:
if not recommendations_df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    cat_counts = recommendations_df["predicted_category"].value_counts()
    sns.barplot(x=cat_counts.values, y=cat_counts.index, ax=ax1, palette="viridis")
    ax1.set_title("Top Forecasted Categories for Recommendations")
    ax1.set_xlabel("Session Count")
    if "severity" in recommendations_df.columns:
        sev_counts = recommendations_df["severity"].value_counts()
        colors = ["#ff4b4b", "#ffa726", "#66bb6a"]
        ax2.pie(sev_counts.values, labels=sev_counts.index, autopct="%1.1f%%", colors=colors)
        ax2.set_title("Recommendation Severity Distribution")
    plt.tight_layout()
    plt.show()

## 4. Sample Recommendation Cards Inspection

In [ ]:
sample_session = recommendations_df["target_session_id"].iloc[0]
sample_recs = recommendations_df[recommendations_df["target_session_id"] == sample_session]

print(f"=== Recommendations for Session #{sample_session} ===")
for idx, row in sample_recs.iterrows():
    cat = row["recommended_category"]
    score = row.get("score", 0)
    print(f"Rank #{row.get('rank', idx+1)}: Category '{cat}' (Score: {score:.3f})")
    print(f"  Rationale: {row.get('rationale', 'N/A')}")
    print(f"  Evidence: {row.get('evidence', 'N/A')}\n")